<a href="https://colab.research.google.com/github/Dhanush-sai-reddy/Dhanush-sai-reddy/blob/main/mluci%20phishing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import re
import pickle
import warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import kagglehub

warnings.filterwarnings("ignore")

# ============================
# 1. LOAD UCI DATASET
# ============================

def load_dataset():
    """Download and load the UCI phishing dataset from KaggleHub"""
    print("Downloading UCI dataset from KaggleHub...")
    path = kagglehub.dataset_download("isatish/phishing-dataset-uci-ml-csv")
    csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

    if not csv_files:
        raise Exception("No CSV found in downloaded dataset folder.")

    df = pd.read_csv(os.path.join(path, csv_files[0]))
    print(f"Dataset Loaded: {df.shape}")
    print(f"Class distribution:\n{df['Result'].value_counts()}")
    print(f"-1: Legitimate, 1: Phishing")
    return df


# ============================
# 2. TRAIN MODEL
# ============================

def train_model(df):
    """Train an ensemble model on the dataset"""
    print("\n" + "="*50)
    print("TRAINING MODEL")
    print("="*50)

    # Prepare features and target
    X = df.drop(["Result", "id"], axis=1)  # Exclude 'id' column
    y = (df["Result"] == 1).astype(int)    # Convert {-1,1} → {0,1} where 1=Phishing

    print(f"Features: {X.shape[1]}")
    print(f"Samples: {X.shape[0]}")
    print(f"Phishing samples: {y.sum()} ({y.sum()/len(y)*100:.1f}%)")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train Random Forest
    print("\nTraining RandomForest...")
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)
    print(f"RandomForest Accuracy: {rf_acc:.4f}")

    # Train XGBoost
    print("\nTraining XGBoost...")
    xgb = XGBClassifier(
        n_estimators=250,
        max_depth=7,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)
    xgb_acc = accuracy_score(y_test, xgb_pred)
    print(f"XGBoost Accuracy: {xgb_acc:.4f}")

    # Train Ensemble (Voting Classifier)
    print("\nTraining Ensemble Model...")
    voting = VotingClassifier(
        estimators=[("rf", rf), ("xgb", xgb)],
        voting="soft"
    )
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)
    voting_acc = accuracy_score(y_test, voting_pred)
    print(f"Ensemble Accuracy: {voting_acc:.4f}")

    # Detailed report
    print("\n" + "="*50)
    print("ENSEMBLE MODEL PERFORMANCE")
    print("="*50)
    print(classification_report(y_test, voting_pred,
                               target_names=["Legitimate", "Phishing"]))

    return voting, X.columns.tolist()


# ============================
# 3. SAVE/LOAD MODEL
# ============================

def save_model(model, filename="phishing_model.pkl"):
    """Save trained model to disk"""
    with open(filename, "wb") as f:
        pickle.dump(model, f)
    print(f"\nModel saved as: {filename}")
    return filename

def load_saved_model(filename="phishing_model.pkl"):
    """Load saved model from disk"""
    with open(filename, "rb") as f:
        model = pickle.load(f)
    print(f"Model loaded from: {filename}")
    return model


# ============================
# 4. FEATURE EXTRACTION (URL + HTML/JS)
# ============================

UCI_FEATURES = [
    "having_IP_Address", "URL_Length", "Shortining_Service", "having_At_Symbol",
    "double_slash_redirecting", "Prefix_Suffix", "having_Sub_Domain",
    "SSLfinal_State", "Domain_registeration_length", "Favicon", "port",
    "HTTPS_token", "Request_URL", "URL_of_Anchor", "Links_in_tags", "SFH",
    "Submitting_to_email", "Abnormal_URL", "Redirect", "on_mouseover",
    "RightClick", "popUpWidnow", "Iframe", "age_of_domain", "DNSRecord",
    "web_traffic", "Page_Rank", "Google_Index", "Links_pointing_to_page",
    "Statistical_report"
]

def extract_features(url):
    """
    Extract 30 features from a URL based on UCI phishing dataset specification
    Returns: Dictionary of feature names and values {-1, 0, 1}
    """
    features = {}

    # Parse URL
    p = urlparse(url)
    domain = p.netloc
    dom = domain.replace("www.", "").lower()

    # ========== URL-BASED FEATURES ==========

    # 1. Having IP Address
    ip_pattern = r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$'
    features["having_IP_Address"] = 1 if re.match(ip_pattern, domain) else -1

    # 2. URL Length
    url_length = len(url)
    if url_length < 54:
        features["URL_Length"] = -1  # Legitimate
    elif 54 <= url_length <= 75:
        features["URL_Length"] = 0   # Suspicious
    else:
        features["URL_Length"] = 1   # Phishing

    # 3. Shortening Service
    shorteners = r'(bit\.ly|goo\.gl|tinyurl|is\.gd|t\.co|ow\.ly|buff\.ly|adf\.ly|bitly\.com|shorte\.st)'
    features["Shortining_Service"] = 1 if re.search(shorteners, url, re.IGNORECASE) else -1

    # 4. Having @ Symbol
    features["having_At_Symbol"] = 1 if "@" in url else -1

    # 5. Double Slash Redirecting
    features["double_slash_redirecting"] = 1 if url.count("//") > 1 else -1

    # 6. Prefix/Suffix
    features["Prefix_Suffix"] = 1 if "-" in domain else -1

    # 7. Having Sub Domain
    subdomains = domain.count(".")
    if subdomains == 1:
        features["having_Sub_Domain"] = -1  # Legitimate (e.g., example.com)
    elif subdomains == 2:
        features["having_Sub_Domain"] = 0   # Suspicious (e.g., www.example.com)
    else:
        features["having_Sub_Domain"] = 1   # Phishing (e.g., www.sub.example.com)

    # 8. SSL Final State
    features["SSLfinal_State"] = 1 if p.scheme == "https" else -1

    # 9. HTTPS Token in Domain
    features["HTTPS_token"] = 1 if "https" in domain.lower() else -1

    # 10. Port
    features["port"] = 1 if p.port not in [80, 443, None] else -1

    # ========== PLACEHOLDER FEATURES (Can't verify from URL alone) ==========
    # These would require external APIs/databases
    placeholder_features = [
        "Domain_registeration_length", "Favicon", "age_of_domain", "DNSRecord",
        "web_traffic", "Page_Rank", "Google_Index", "Links_pointing_to_page",
        "Statistical_report"
    ]
    for pf in placeholder_features:
        features[pf] = -1  # Default to legitimate (can't verify)

    # ========== HTML-BASED FEATURES ==========
    html_features = [
        "Request_URL", "URL_of_Anchor", "Links_in_tags", "SFH",
        "Submitting_to_email", "Abnormal_URL", "Redirect", "on_mouseover",
        "RightClick", "popUpWidnow", "Iframe"
    ]

    # Initialize HTML features to neutral (0) - will update if HTML is fetched
    for hf in html_features:
        features[hf] = 0

    try:
        # Try to fetch HTML with timeout
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, timeout=5, headers=headers, verify=False)
        response.raise_for_status()
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')

        # Extract HTML elements
        anchors = soup.find_all('a', href=True)
        imgs = soup.find_all('img')
        scripts = soup.find_all('script')
        links = soup.find_all('link')
        iframes = soup.find_all('iframe')
        forms = soup.find_all('form')

        # 11. Request URL (external objects)
        total_objects = len(imgs) + len(scripts)
        if total_objects > 0:
            external_objects = 0
            for tag in imgs + scripts:
                src = tag.get('src', '')
                if src and src.startswith('http') and dom not in src:
                    external_objects += 1
            external_ratio = external_objects / total_objects
            if external_ratio < 0.22:
                features["Request_URL"] = -1
            elif external_ratio <= 0.61:
                features["Request_URL"] = 0
            else:
                features["Request_URL"] = 1
        else:
            features["Request_URL"] = 0

        # 12. URL of Anchor
        if anchors:
            external_anchors = 0
            for a in anchors:
                href = a.get('href', '')
                if href and href.startswith('http') and dom not in href:
                    external_anchors += 1
            external_ratio = external_anchors / len(anchors)
            if external_ratio < 0.31:
                features["URL_of_Anchor"] = -1
            elif external_ratio <= 0.67:
                features["URL_of_Anchor"] = 0
            else:
                features["URL_of_Anchor"] = 1
        else:
            features["URL_of_Anchor"] = 0

        # 13. Links in Tags (Meta, Script, Link tags)
        meta_tags = soup.find_all('meta')
        all_tags = scripts + links + meta_tags
        if all_tags:
            external_links = 0
            for tag in all_tags:
                src = tag.get('src', '') or tag.get('href', '') or tag.get('content', '')
                if src and src.startswith('http') and dom not in src:
                    external_links += 1
            external_ratio = external_links / len(all_tags)
            if external_ratio < 0.17:
                features["Links_in_tags"] = -1
            elif external_ratio <= 0.81:
                features["Links_in_tags"] = 0
            else:
                features["Links_in_tags"] = 1
        else:
            features["Links_in_tags"] = 0

        # 14. SFH (Server Form Handler)
        if not forms:
            features["SFH"] = -1
        else:
            form_action = forms[0].get('action', '').lower()
            if not form_action or form_action == 'about:blank':
                features["SFH"] = 1  # Phishing
            elif form_action.startswith('http') and dom not in form_action:
                features["SFH"] = 0  # Suspicious
            else:
                features["SFH"] = -1  # Legitimate

        # 15. Submitting to Email
        features["Submitting_to_email"] = 1 if 'mailto:' in html.lower() else -1

        # 16. Abnormal URL (hostname in page content)
        features["Abnormal_URL"] = 1 if dom not in html.lower() else -1

        # 17. Redirect
        redirect_patterns = [
            'window.location', 'window.open', 'location.href',
            'location.replace', 'http-equiv="refresh"'
        ]
        has_redirect = any(pattern in html.lower() for pattern in redirect_patterns)
        features["Redirect"] = 1 if has_redirect else -1

        # 18. On Mouse Over
        features["on_mouseover"] = 1 if 'onmouseover' in html.lower() else -1

        # 19. Right Click Disabled
        features["RightClick"] = 1 if 'event.button==2' in html or 'contextmenu' in html.lower() else -1

        # 20. Popup Window
        features["popUpWidnow"] = 1 if 'window.open' in html or 'alert(' in html else -1

        # 21. IFrame
        features["Iframe"] = 1 if iframes else -1

    except Exception as e:
        # HTML fetch failed - keep default values (0 for HTML features)
        pass

    # Ensure all features are present
    for feature in UCI_FEATURES:
        if feature not in features:
            features[feature] = 0

    return features


# ============================
# 5. PREDICT WITH RULE-BASED OVERRIDES
# ============================

def predict_url(url, model, feature_cols, use_rules=True):
    """
    Predict if URL is phishing with optional rule-based overrides
    Returns: (prediction, probability, features_dict)
    """
    # Extract features
    features = extract_features(url)

    # Create dataframe for model prediction
    df = pd.DataFrame([features])[feature_cols]

    # Get model prediction
    prob = model.predict_proba(df)[0][1]  # Probability of being phishing
    pred = model.predict(df)[0]  # 1=Phishing, 0=Legitimate

    # ========== RULE-BASED OVERRIDES ==========
    if use_rules:
        # Rule 1: IP Address in URL is highly suspicious
        if features["having_IP_Address"] == 1:
            pred = 1  # Force phishing
            prob = max(prob, 0.9)  # High confidence

        # Rule 2: Shortened URLs are suspicious
        elif features["Shortining_Service"] == 1:
            pred = 1  # Force phishing
            prob = max(prob, 0.8)

        # Rule 3: Very long URLs are suspicious
        elif features["URL_Length"] == 1:
            pred = 1
            prob = max(prob, 0.7)

        # Rule 4: Multiple strong legitimate indicators
        legit_indicators = sum(1 for k, v in features.items() if v == -1)
        phishing_indicators = sum(1 for k, v in features.items() if v == 1)

        if legit_indicators >= 20 and phishing_indicators <= 2:
            pred = 0  # Force legitimate
            prob = min(prob, 0.1)

    return pred, prob, features


# ============================
# 6. DEBUG & ANALYSIS TOOLS
# ============================

def analyze_features(features):
    """Analyze and display feature breakdown"""
    print("\n" + "="*60)
    print("FEATURE ANALYSIS")
    print("="*60)

    phishing_features = [k for k, v in features.items() if v == 1]
    suspicious_features = [k for k, v in features.items() if v == 0]
    legitimate_features = [k for k, v in features.items() if v == -1]

    print(f"\nPhishing Indicators ({len(phishing_features)}):")
    print(", ".join(phishing_features) if phishing_features else "None")

    print(f"\nSuspicious/Neutral ({len(suspicious_features)}):")
    print(", ".join(suspicious_features[:10]) + ("..." if len(suspicious_features) > 10 else ""))

    print(f"\nLegitimate Indicators ({len(legitimate_features)}):")
    print(", ".join(legitimate_features[:10]) + ("..." if len(legitimate_features) > 10 else ""))

    return len(phishing_features), len(suspicious_features), len(legitimate_features)


def test_urls(urls, model, feature_cols):
    """Test multiple URLs and display results"""
    print("\n" + "="*60)
    print("URL TESTING RESULTS")
    print("="*60)

    results = []
    for url in urls:
        pred, prob, features = predict_url(url, model, feature_cols)

        print(f"\n{'='*40}")
        print(f"URL: {url}")
        print(f"{'='*40}")
        print(f"PREDICTION: {'🚨 PHISHING' if pred == 1 else '✅ LEGITIMATE'}")
        print(f"CONFIDENCE: {prob:.1%}")

        # Show top features
        phishing, suspicious, legitimate = analyze_features(features)

        # Store results
        results.append({
            'url': url,
            'prediction': 'Phishing' if pred == 1 else 'Legitimate',
            'confidence': prob,
            'phishing_indicators': phishing,
            'legitimate_indicators': legitimate
        })

    return results


# ============================
# 7. MAIN EXECUTION
# ============================

if __name__ == "__main__":
    print("="*60)
    print("PHISHING URL DETECTOR")
    print("="*60)

    # Option 1: Train new model
    train_new = input("\nTrain new model? (y/n): ").lower().strip() == 'y'

    if train_new:
        # Load dataset and train
        df = load_dataset()
        model, feature_cols = train_model(df)

        # Save model
        model_file = save_model(model, "phishing_detector_model.pkl")

        # Save feature columns for later use
        with open("feature_columns.pkl", "wb") as f:
            pickle.dump(feature_cols, f)
        print("Feature columns saved.")

    else:
        # Load existing model
        try:
            model = load_saved_model("phishing_detector_model.pkl")
            with open("feature_columns.pkl", "rb") as f:
                feature_cols = pickle.load(f)
            print(f"Loaded {len(feature_cols)} features")
        except FileNotFoundError:
            print("No saved model found. Training new model...")
            df = load_dataset()
            model, feature_cols = train_model(df)
            save_model(model, "phishing_detector_model.pkl")

    # Test with example URLs
    test_urls_list = [
        "https://google.com",
        "http://198.54.23.11/login/update",  # IP address URL
        "https://paypal-security-alert.com/verify",  # Phishing-like
        "http://bit.ly/2fSdq",  # Shortened URL
        "https://github.com",
        "http://free-gift-cards-now.com/claim",  # Likely phishing
        "https://www.amazon.com",
        "http://192.168.1.100:8080/admin",  # Local IP
        "https://www.paypal.com.us.security.verify-account.com",  # Suspicious
        "https://www.wikipedia.org"
    ]

    # Test the URLs
    results = test_urls(test_urls_list, model, feature_cols)

    # Summary
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    phishing_count = sum(1 for r in results if r['prediction'] == 'Phishing')
    print(f"\nTotal URLs tested: {len(results)}")
    print(f"Phishing detected: {phishing_count}")
    print(f"Legitimate: {len(results) - phishing_count}")

    # Interactive mode
    while True:
        print("\n" + "-"*40)
        user_url = input("\nEnter URL to check (or 'quit' to exit): ").strip()

        if user_url.lower() in ['quit', 'exit', 'q']:
            print("Exiting...")
            break

        if not user_url.startswith(('http://', 'https://')):
            user_url = 'http://' + user_url

        try:
            pred, prob, features = predict_url(user_url, model, feature_cols)

            print(f"\n{'='*50}")
            print(f"RESULT: {'🚨 PHISHING' if pred == 1 else '✅ LEGITIMATE'}")
            print(f"Confidence: {prob:.1%}")
            print(f"{'='*50}")

            # Show key indicators
            print("\nKEY INDICATORS:")
            if features["having_IP_Address"] == 1:
                print("  ⚠️  IP Address in URL")
            if features["Shortining_Service"] == 1:
                print("  ⚠️  URL Shortening Service")
            if features["SSLfinal_State"] == -1:
                print("  ⚠️  No HTTPS (HTTP only)")
            if features["having_At_Symbol"] == 1:
                print("  ⚠️  @ Symbol in URL")

            # Ask for detailed analysis
            if input("\nShow detailed analysis? (y/n): ").lower() == 'y':
                analyze_features(features)

        except Exception as e:
            print(f"Error analyzing URL: {e}")

PHISHING URL DETECTOR

Train new model? (y/n): y


100%|██████████| 110k/110k [00:00<00:00, 552kB/s]

Extracting files...


Dataset Loaded: (11055, 32)
Class distribution:
Result
 1    6157
-1    4898
Name: count, dtype: int64
-1: Legitimate, 1: Phishing

TRAINING MODEL
Features: 30
Samples: 11055
Phishing samples: 6157 (55.7%)

Training RandomForest...
RandomForest Accuracy: 0.9742

Training XGBoost...
XGBoost Accuracy: 0.9751

Training Ensemble Model...
Ensemble Accuracy: 0.9774

ENSEMBLE MODEL PERFORMANCE
              precision    recall  f1-score   support

  Legitimate       0.98      0.97      0.97       980
    Phishing       0.97      0.99      0.98      1231

    accuracy                           0.98      2211
   macro avg       0.98      0.98      0.98      2211
weighted avg       0.98      0.98      0.98      2211


Model saved as: phishing_detector_model.pkl
Feature columns saved.

URL TESTING RESULTS

URL: https://google.com
PREDICTION: ✅ LEGITIMATE
CONFIDENCE: 14.9%

FEATURE ANALYSIS

Phishing Indicators (3):
SSLfinal_State, Submitting_to_email, Redirect

Suspicious/Neutral (0):


Legitimat

In [ ]:
import os
import re
import pickle
import warnings
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import kagglehub

warnings.filterwarnings("ignore")

# ============================
# 1. LOAD UCI DATASET
# ============================

def load_dataset():
    """Download and load the UCI phishing dataset from KaggleHub"""
    print("Downloading UCI dataset from KaggleHub...")
    path = kagglehub.dataset_download("isatish/phishing-dataset-uci-ml-csv")
    csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

    if not csv_files:
        raise Exception("No CSV found in downloaded dataset folder.")

    df = pd.read_csv(os.path.join(path, csv_files[0]))
    print(f"Dataset Loaded: {df.shape}")
    print(f"Class distribution:\n{df['Result'].value_counts()}")
    print(f"-1: Legitimate, 1: Phishing")
    return df


# ============================
# 2. TRAIN MODEL (KEEP EXACTLY THE SAME)
# ============================

def train_model(df):
    """Train an ensemble model on the dataset"""
    print("\n" + "="*50)
    print("TRAINING MODEL")
    print("="*50)

    # Prepare features and target
    X = df.drop(["Result", "id"], axis=1)  # Exclude 'id' column
    y = (df["Result"] == 1).astype(int)    # Convert {-1,1} → {0,1} where 1=Phishing

    print(f"Features: {X.shape[1]}")
    print(f"Samples: {X.shape[0]}")
    print(f"Phishing samples: {y.sum()} ({y.sum()/len(y)*100:.1f}%)")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train Random Forest
    print("\nTraining RandomForest...")
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)
    print(f"RandomForest Accuracy: {rf_acc:.4f}")

    # Train XGBoost
    print("\nTraining XGBoost...")
    xgb = XGBClassifier(
        n_estimators=250,
        max_depth=7,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)
    xgb_acc = accuracy_score(y_test, xgb_pred)
    print(f"XGBoost Accuracy: {xgb_acc:.4f}")

    # Train Ensemble (Voting Classifier)
    print("\nTraining Ensemble Model...")
    voting = VotingClassifier(
        estimators=[("rf", rf), ("xgb", xgb)],
        voting="soft"
    )
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)
    voting_acc = accuracy_score(y_test, voting_pred)
    print(f"Ensemble Accuracy: {voting_acc:.4f}")

    # Detailed report
    print("\n" + "="*50)
    print("ENSEMBLE MODEL PERFORMANCE")
    print("="*50)
    print(classification_report(y_test, voting_pred,
                               target_names=["Legitimate", "Phishing"]))

    # Confusion Matrix
    cm = confusion_matrix(y_test, voting_pred)
    print("\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"                Legit  Phishing")
    print(f"Actual Legit     {cm[0,0]:5d}   {cm[0,1]:5d}")
    print(f"Actual Phishing  {cm[1,0]:5d}   {cm[1,1]:5d}")

    return voting, X.columns.tolist()


# ============================
# 3. SAVE/LOAD MODEL
# ============================

def save_model(model, filename="phishing_model.pkl"):
    """Save trained model to disk"""
    with open(filename, "wb") as f:
        pickle.dump(model, f)
    print(f"\nModel saved as: {filename}")
    return filename

def load_saved_model(filename="phishing_model.pkl"):
    """Load saved model from disk"""
    with open(filename, "rb") as f:
        model = pickle.load(f)
    print(f"Model loaded from: {filename}")
    return model


# ============================
# 4. FIXED FEATURE EXTRACTION
# ============================

UCI_FEATURES = [
    "having_IP_Address", "URL_Length", "Shortining_Service", "having_At_Symbol",
    "double_slash_redirecting", "Prefix_Suffix", "having_Sub_Domain",
    "SSLfinal_State", "Domain_registeration_length", "Favicon", "port",
    "HTTPS_token", "Request_URL", "URL_of_Anchor", "Links_in_tags", "SFH",
    "Submitting_to_email", "Abnormal_URL", "Redirect", "on_mouseover",
    "RightClick", "popUpWidnow", "Iframe", "age_of_domain", "DNSRecord",
    "web_traffic", "Page_Rank", "Google_Index", "Links_pointing_to_page",
    "Statistical_report"
]

def extract_features_corrected(url):
    """
    FIXED: Extract 30 features from a URL with correct logic
    Key fixes:
    1. SSLfinal_State: -1 for HTTPS (legitimate), 1 for HTTP (phishing)
    2. Better URL shortening detection
    3. Improved subdomain counting
    4. Better IP address detection
    """
    features = {}

    # Parse URL
    p = urlparse(url)
    domain = p.netloc
    dom = domain.replace("www.", "").lower()

    # Clean domain for analysis
    if domain.startswith('www.'):
        domain = domain[4:]

    # ========== URL-BASED FEATURES ==========

    # 1. Having IP Address (1=Phishing, -1=Legitimate)
    ip_pattern = r'^(\d{1,3}\.){3}\d{1,3}(:\d+)?$'
    features["having_IP_Address"] = 1 if re.match(ip_pattern, domain) else -1

    # 2. URL Length (1=Phishing for long, -1=Legitimate for short)
    url_length = len(url)
    if url_length < 54:
        features["URL_Length"] = -1  # Legitimate
    elif 54 <= url_length <= 75:
        features["URL_Length"] = 0   # Suspicious
    else:
        features["URL_Length"] = 1   # Phishing

    # 3. Shortening Service - FIXED: Only match real shorteners in domain
    shorteners = [
        'bit.ly', 'goo.gl', 'tinyurl.com', 'is.gd', 't.co', 'ow.ly', 'buff.ly',
        'adf.ly', 'bitly.com', 'shorte.st', 'tiny.cc', 'short.to', 'tr.im',
        'qr.ae', 'v.gd', 'cur.lv', 'clickmeter.com', 'soo.gd', 'ity.im',
        'q.gs', 'po.st', 'bc.vc', 'twitthis.com', 'u.to', 'j.mp', 'buzurl.com',
        'cutt.us', 'u.bb', 'yourls.org', 'x.co'
    ]

    is_shortener = any(shortener in domain.lower() for shortener in shorteners)
    features["Shortining_Service"] = 1 if is_shortener else -1

    # 4. Having @ Symbol (1=Phishing, -1=Legitimate)
    features["having_At_Symbol"] = 1 if "@" in url else -1

    # 5. Double Slash Redirecting (1=Phishing, -1=Legitimate)
    # Count "//" after the initial http:// or https://
    path_part = url.split('://', 1)[-1] if '://' in url else url
    features["double_slash_redirecting"] = 1 if path_part.count('//') > 0 else -1

    # 6. Prefix/Suffix (1=Phishing for hyphens, -1=Legitimate)
    features["Prefix_Suffix"] = 1 if "-" in domain else -1

    # 7. Having Sub Domain - FIXED: Better counting
    # Remove www. and count dots
    clean_domain = domain.replace('www.', '')
    subdomain_count = clean_domain.count('.')

    if subdomain_count == 0:
        features["having_Sub_Domain"] = -1  # Legitimate (e.g., example.com)
    elif subdomain_count == 1:
        features["having_Sub_Domain"] = 0   # Suspicious (e.g., www.example.com)
    else:
        features["having_Sub_Domain"] = 1   # Phishing (e.g., sub.sub.example.com)

    # 8. SSL Final State - FIXED: HTTPS is LEGITIMATE (-1), HTTP is PHISHING (1)
    if p.scheme == "https":
        features["SSLfinal_State"] = -1  # HTTPS is GOOD
    else:
        features["SSLfinal_State"] = 1   # HTTP is BAD

    # 9. HTTPS Token in Domain (1=Phishing if "https" appears in domain)
    features["HTTPS_token"] = 1 if "https" in domain.lower() else -1

    # 10. Port (1=Phishing for non-standard ports)
    features["port"] = 1 if p.port not in [80, 443, None] else -1

    # ========== PLACEHOLDER FEATURES ==========
    placeholder_features = [
        "Domain_registeration_length", "Favicon", "age_of_domain", "DNSRecord",
        "web_traffic", "Page_Rank", "Google_Index", "Links_pointing_to_page",
        "Statistical_report"
    ]
    for pf in placeholder_features:
        features[pf] = -1  # Default to legitimate

    # ========== HTML-BASED FEATURES ==========
    html_features = [
        "Request_URL", "URL_of_Anchor", "Links_in_tags", "SFH",
        "Submitting_to_email", "Abnormal_URL", "Redirect", "on_mouseover",
        "RightClick", "popUpWidnow", "Iframe"
    ]

    for hf in html_features:
        features[hf] = 0  # Neutral default

    # Try to fetch HTML
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, timeout=5, headers=headers, verify=False)
        response.raise_for_status()
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')

        # Extract HTML elements
        anchors = soup.find_all('a', href=True)
        imgs = soup.find_all('img')
        scripts = soup.find_all('script')
        links = soup.find_all('link')
        iframes = soup.find_all('iframe')
        forms = soup.find_all('form')

        # 11. Request URL (external objects)
        total_objects = len(imgs) + len(scripts)
        if total_objects > 0:
            external_objects = 0
            for tag in imgs + scripts:
                src = tag.get('src', '')
                if src and src.startswith('http') and dom not in src:
                    external_objects += 1
            external_ratio = external_objects / total_objects
            if external_ratio < 0.22:
                features["Request_URL"] = -1
            elif external_ratio <= 0.61:
                features["Request_URL"] = 0
            else:
                features["Request_URL"] = 1
        else:
            features["Request_URL"] = 0

        # 12. URL of Anchor
        if anchors:
            external_anchors = 0
            for a in anchors:
                href = a.get('href', '')
                if href and href.startswith('http') and dom not in href:
                    external_anchors += 1
            external_ratio = external_anchors / len(anchors)
            if external_ratio < 0.31:
                features["URL_of_Anchor"] = -1
            elif external_ratio <= 0.67:
                features["URL_of_Anchor"] = 0
            else:
                features["URL_of_Anchor"] = 1
        else:
            features["URL_of_Anchor"] = 0

        # 13. Links in Tags
        meta_tags = soup.find_all('meta')
        all_tags = scripts + links + meta_tags
        if all_tags:
            external_links = 0
            for tag in all_tags:
                src = tag.get('src', '') or tag.get('href', '') or tag.get('content', '')
                if src and src.startswith('http') and dom not in src:
                    external_links += 1
            external_ratio = external_links / len(all_tags)
            if external_ratio < 0.17:
                features["Links_in_tags"] = -1
            elif external_ratio <= 0.81:
                features["Links_in_tags"] = 0
            else:
                features["Links_in_tags"] = 1
        else:
            features["Links_in_tags"] = 0

        # 14. SFH (Server Form Handler)
        if not forms:
            features["SFH"] = -1
        else:
            form_action = forms[0].get('action', '').lower()
            if not form_action or form_action == 'about:blank':
                features["SFH"] = 1  # Phishing
            elif form_action.startswith('http') and dom not in form_action:
                features["SFH"] = 0  # Suspicious
            else:
                features["SFH"] = -1  # Legitimate

        # 15. Submitting to Email
        features["Submitting_to_email"] = 1 if 'mailto:' in html.lower() else -1

        # 16. Abnormal URL (hostname in page content)
        features["Abnormal_URL"] = 1 if dom not in html.lower() else -1

        # 17. Redirect
        redirect_patterns = [
            'window.location', 'window.open', 'location.href',
            'location.replace', 'http-equiv="refresh"'
        ]
        has_redirect = any(pattern in html.lower() for pattern in redirect_patterns)
        features["Redirect"] = 1 if has_redirect else -1

        # 18. On Mouse Over
        features["on_mouseover"] = 1 if 'onmouseover' in html.lower() else -1

        # 19. Right Click Disabled
        features["RightClick"] = 1 if 'event.button==2' in html or 'contextmenu' in html.lower() else -1

        # 20. Popup Window
        features["popUpWidnow"] = 1 if 'window.open' in html or 'alert(' in html else -1

        # 21. IFrame
        features["Iframe"] = 1 if iframes else -1

    except Exception as e:
        # HTML fetch failed - keep default values
        pass

    # Ensure all features are present
    for feature in UCI_FEATURES:
        if feature not in features:
            features[feature] = 0

    return features


# ============================
# 5. IMPROVED PREDICTION WITH SMART RULES
# ============================

def predict_url_improved(url, model, feature_cols, use_rules=True):
    """
    Improved prediction with smart rule-based overrides
    """
    # Extract features with corrected logic
    features = extract_features_corrected(url)

    # Create dataframe for model prediction
    df = pd.DataFrame([features])[feature_cols]

    # Get model prediction
    prob = model.predict_proba(df)[0][1]  # Probability of being phishing
    pred = model.predict(df)[0]  # 1=Phishing, 0=Legitimate

    # ========== SMART RULE-BASED OVERRIDES ==========
    if use_rules:
        # Parse URL for additional checks
        p = urlparse(url)
        domain = p.netloc.lower()

        # Whitelist of known legitimate domains (partial matching)
        legit_domains = [
            'google.com', 'github.com', 'amazon.com', 'wikipedia.org',
            'microsoft.com', 'apple.com', 'netflix.com', 'twitter.com',
            'facebook.com', 'instagram.com', 'linkedin.com', 'chatgpt.com',
            'openai.com', 'deepseek.com', 'anthropic.com', 'yahoo.com',
            'reddit.com', 'stackoverflow.com', 'medium.com', 'quora.com'
        ]

        # Check if domain contains any known legit domain
        is_known_legit = any(legit in domain for legit in legit_domains)

        if is_known_legit:
            # Known legitimate site - override to legitimate with high confidence
            pred = 0
            prob = 0.05
            return pred, prob, features

        # Blacklist patterns for known phishing
        phishing_patterns = [
            'paypal-security', 'bank-security', 'login-verify',
            'account-update', 'secure-verify', 'password-reset'
        ]

        if any(pattern in url.lower() for pattern in phishing_patterns):
            pred = 1
            prob = 0.95
            return pred, prob, features

        # Rule 1: Strong phishing indicators (any one triggers phishing)
        strong_phishing = [
            features["having_IP_Address"] == 1,  # IP in URL
            features["Shortining_Service"] == 1,  # URL shortener
            features["having_At_Symbol"] == 1,    # @ in URL
        ]

        if any(strong_phishing):
            pred = 1
            prob = max(prob, 0.85)

        # Rule 2: HTTP without HTTPS (very suspicious)
        elif features["SSLfinal_State"] == 1:  # HTTP is BAD
            # Check if it's a sensitive page (login, account, etc.)
            sensitive_terms = ['login', 'signin', 'account', 'bank', 'pay', 'secure']
            path = p.path.lower()
            if any(term in path for term in sensitive_terms):
                pred = 1
                prob = max(prob, 0.90)

        # Rule 3: Long URL with multiple suspicious features
        elif (features["URL_Length"] == 1 and
              (features["Prefix_Suffix"] == 1 or
               features["having_Sub_Domain"] == 1 or
               features["double_slash_redirecting"] == 1)):
            pred = 1
            prob = max(prob, 0.75)

        # Rule 4: Strong legitimate indicators (override to legitimate)
        strong_legit = [
            features["SSLfinal_State"] == -1,      # HTTPS
            features["having_IP_Address"] == -1,   # No IP
            features["Shortining_Service"] == -1,  # No shortener
            features["having_At_Symbol"] == -1,    # No @
            features["Prefix_Suffix"] == -1,       # No hyphens
        ]

        if sum(strong_legit) >= 4:  # 4 out of 5 strong legit indicators
            pred = 0
            prob = min(prob, 0.15)

        # Rule 5: Very short legitimate URLs
        elif features["URL_Length"] == -1 and len(url) < 40:
            pred = 0
            prob = min(prob, 0.10)

    return pred, prob, features


# ============================
# 6. ENHANCED DEBUG & ANALYSIS TOOLS
# ============================

def analyze_features_detailed(features):
    """Detailed feature analysis with color coding"""
    print("\n" + "="*60)
    print("DETAILED FEATURE ANALYSIS")
    print("="*60)

    phishing_features = [k for k, v in features.items() if v == 1]
    suspicious_features = [k for k, v in features.items() if v == 0]
    legitimate_features = [k for k, v in features.items() if v == -1]

    print(f"\n🚨 Phishing Indicators ({len(phishing_features)}):")
    if phishing_features:
        for pf in phishing_features:
            print(f"  ⚠️  {pf}")
    else:
        print("  ✅ None")

    print(f"\n⚠️  Suspicious/Neutral ({len(suspicious_features)}):")
    if suspicious_features:
        for sf in suspicious_features[:5]:  # Show only top 5
            print(f"  ⚠️  {sf}")
        if len(suspicious_features) > 5:
            print(f"  ... and {len(suspicious_features) - 5} more")
    else:
        print("  ✅ None")

    print(f"\n✅ Legitimate Indicators ({len(legitimate_features)}):")
    if legitimate_features:
        # Show key legitimate indicators first
        key_legit = [f for f in legitimate_features if f in [
            "SSLfinal_State", "having_IP_Address", "Shortining_Service",
            "having_At_Symbol", "Prefix_Suffix", "URL_Length"
        ]]

        for lf in key_legit:
            print(f"  ✅ {lf}")

        if len(legitimate_features) > len(key_legit):
            print(f"  ✅ ... and {len(legitimate_features) - len(key_legit)} more indicators")
    else:
        print("  ⚠️  None")

    return len(phishing_features), len(suspicious_features), len(legitimate_features)


def test_urls_comprehensive(urls, model, feature_cols):
    """Comprehensive URL testing with detailed analysis"""
    print("\n" + "="*60)
    print("COMPREHENSIVE URL TESTING")
    print("="*60)

    results = []
    for i, url in enumerate(urls, 1):
        pred, prob, features = predict_url_improved(url, model, feature_cols)

        print(f"\n{'='*40}")
        print(f"TEST #{i}: {url}")
        print(f"{'='*40}")
        print(f"PREDICTION: {'🚨 PHISHING' if pred == 1 else '✅ LEGITIMATE'}")
        print(f"CONFIDENCE: {prob:.1%}")

        # Show feature breakdown
        phishing, suspicious, legitimate = analyze_features_detailed(features)

        # Additional analysis
        p = urlparse(url)
        print(f"\n📊 URL Analysis:")
        print(f"  Domain: {p.netloc}")
        print(f"  Protocol: {p.scheme}")
        print(f"  Path: {p.path[:50]}{'...' if len(p.path) > 50 else ''}")

        # Store results
        results.append({
            'url': url,
            'prediction': 'Phishing' if pred == 1 else 'Legitimate',
            'confidence': prob,
            'phishing_indicators': phishing,
            'suspicious_indicators': suspicious,
            'legitimate_indicators': legitimate,
            'domain': p.netloc
        })

    return results


# ============================
# 7. MAIN EXECUTION WITH IMPROVED FEATURES
# ============================

if __name__ == "__main__":
    print("="*70)
    print("PHISHING URL DETECTOR - IMPROVED VERSION")
    print("="*70)

    # Option 1: Train new model
    train_new = input("\nTrain new model? (y/n): ").lower().strip() == 'y'

    if train_new:
        # Load dataset and train
        df = load_dataset()
        model, feature_cols = train_model(df)

        # Save model
        model_file = save_model(model, "phishing_detector_improved.pkl")

        # Save feature columns
        with open("feature_columns_improved.pkl", "wb") as f:
            pickle.dump(feature_cols, f)
        print("Feature columns saved.")

    else:
        # Load existing model
        try:
            model = load_saved_model("phishing_detector_improved.pkl")
            with open("feature_columns_improved.pkl", "rb") as f:
                feature_cols = pickle.load(f)
            print(f"Loaded {len(feature_cols)} features")
        except FileNotFoundError:
            print("No saved model found. Training new model...")
            df = load_dataset()
            model, feature_cols = train_model(df)
            save_model(model, "phishing_detector_improved.pkl")
            with open("feature_columns_improved.pkl", "wb") as f:
                pickle.dump(feature_cols, f)

    # Test with comprehensive URL list
    test_urls_list = [
        # Legitimate sites
        "https://google.com",
        "https://github.com",
        "https://www.amazon.com",
        "https://www.wikipedia.org",
        "https://chatgpt.com",
        "https://chat.deepseek.com",
        "https://www.microsoft.com",
        "https://www.apple.com",

        # Phishing sites
        "http://192.168.1.1/login.php",
        "http://paypal-secure-verify.xyz/login",
        "http://bit.ly/2fSdq",
        "http://free-gift-cards-now.com/claim",
        "https://www.paypal.com.us.security.verify-account.com",
        "http://198.54.23.11/login/update",

        # Suspicious/Edge cases
        "http://example.com/login",  # HTTP login page
        "https://secure-bank-login.com",  # Looks suspicious
        "https://paypal-security-alert.com/verify",
        "http://192.168.1.100:8080/admin",
    ]

    # Test the URLs
    results = test_urls_comprehensive(test_urls_list, model, feature_cols)

    # Summary
    print("\n" + "="*70)
    print("TEST SUMMARY")
    print("="*70)

    phishing_count = sum(1 for r in results if r['prediction'] == 'Phishing')
    legitimate_count = len(results) - phishing_count

    print(f"\nTotal URLs tested: {len(results)}")
    print(f"Phishing detected: {phishing_count}")
    print(f"Legitimate detected: {legitimate_count}")

    # Show misclassified examples (if any)
    legit_domains = ['google.com', 'github.com', 'amazon.com', 'wikipedia.org',
                    'microsoft.com', 'apple.com', 'chatgpt.com', 'deepseek.com']

    false_positives = []
    false_negatives = []

    for r in results:
        domain = r['domain'].lower()
        is_actually_legit = any(legit in domain for legit in legit_domains)

        if r['prediction'] == 'Phishing' and is_actually_legit:
            false_positives.append(r['url'])
        elif r['prediction'] == 'Legitimate' and not is_actually_legit:
            # Check if it's clearly phishing
            if 'paypal' in domain or '192.168' in domain or 'bit.ly' in domain:
                false_negatives.append(r['url'])

    if false_positives:
        print(f"\n⚠️  False Positives ({len(false_positives)}):")
        for fp in false_positives:
            print(f"  - {fp}")

    if false_negatives:
        print(f"\n⚠️  False Negatives ({len(false_negatives)}):")
        for fn in false_negatives:
            print(f"  - {fn}")

    # Interactive mode
    while True:
        print("\n" + "-"*50)
        user_url = input("\nEnter URL to check (or 'quit' to exit): ").strip()

        if user_url.lower() in ['quit', 'exit', 'q']:
            print("Exiting...")
            break

        if not user_url.startswith(('http://', 'https://')):
            user_url = 'https://' + user_url

        try:
            pred, prob, features = predict_url_improved(user_url, model, feature_cols)

            print(f"\n{'='*60}")
            print(f"RESULT: {'🚨 PHISHING' if pred == 1 else '✅ LEGITIMATE'}")
            print(f"Confidence: {prob:.1%}")
            print(f"{'='*60}")

            # Quick analysis
            p = urlparse(user_url)
            print(f"\n🔍 Quick Analysis:")
            print(f"  Domain: {p.netloc}")
            print(f"  Protocol: {p.scheme} ({'Secure' if p.scheme == 'https' else 'Insecure'})")

            if p.path:
                print(f"  Path length: {len(p.path)} characters")

            # Show key indicators
            print(f"\n📊 Key Indicators:")

            # Phishing indicators
            phishing_flags = []
            if features.get("having_IP_Address") == 1:
                phishing_flags.append("IP Address in URL")
            if features.get("Shortining_Service") == 1:
                phishing_flags.append("URL Shortening Service")
            if features.get("SSLfinal_State") == 1:
                phishing_flags.append("HTTP (not HTTPS)")
            if features.get("having_At_Symbol") == 1:
                phishing_flags.append("@ Symbol in URL")
            if features.get("Prefix_Suffix") == 1:
                phishing_flags.append("Hyphen in Domain")

            if phishing_flags:
                print(f"  ⚠️  Phishing flags ({len(phishing_flags)}):")
                for flag in phishing_flags:
                    print(f"     • {flag}")
            else:
                print(f"  ✅ No strong phishing flags")

            # Legitimate indicators
            legit_flags = []
            if features.get("SSLfinal_State") == -1:
                legit_flags.append("HTTPS Secure")
            if features.get("having_IP_Address") == -1:
                legit_flags.append("No IP Address")
            if features.get("Shortining_Service") == -1:
                legit_flags.append("No URL Shortening")
            if features.get("having_At_Symbol") == -1:
                legit_flags.append("No @ Symbol")

            if legit_flags:
                print(f"  ✅ Legitimate flags ({len(legit_flags)}):")
                for flag in legit_flags:
                    print(f"     • {flag}")

            # Ask for detailed analysis
            if input("\nShow detailed feature analysis? (y/n): ").lower() == 'y':
                analyze_features_detailed(features)

        except Exception as e:
            print(f"Error analyzing URL: {e}")
            print("Please check the URL format and try again.")

PHISHING URL DETECTOR - IMPROVED VERSION
Using Colab cache for faster access to the 'phishing-dataset-uci-ml-csv' dataset.
Dataset Loaded: (11055, 32)
Class distribution:
Result
 1    6157
-1    4898
Name: count, dtype: int64
-1: Legitimate, 1: Phishing

TRAINING MODEL
Features: 30
Samples: 11055
Phishing samples: 6157 (55.7%)

Training RandomForest...
RandomForest Accuracy: 0.9742

Training XGBoost...
XGBoost Accuracy: 0.9751

Training Ensemble Model...
Ensemble Accuracy: 0.9774

ENSEMBLE MODEL PERFORMANCE
              precision    recall  f1-score   support

  Legitimate       0.98      0.97      0.97       980
    Phishing       0.97      0.99      0.98      1231

    accuracy                           0.98      2211
   macro avg       0.98      0.98      0.98      2211
weighted avg       0.98      0.98      0.98      2211


Confusion Matrix:
                Predicted
                Legit  Phishing
Actual Legit       946      34
Actual Phishing     16    1215

Model saved as: phis

In [ ]:
import os
import re
import pickle
import warnings
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import kagglehub

warnings.filterwarnings("ignore")

# ============================
# 1. LOAD UCI DATASET
# ============================

def load_dataset():
    """Download and load the UCI phishing dataset from KaggleHub"""
    print("Downloading UCI dataset from KaggleHub...")
    path = kagglehub.dataset_download("isatish/phishing-dataset-uci-ml-csv")
    csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

    if not csv_files:
        raise Exception("No CSV found in downloaded dataset folder.")

    df = pd.read_csv(os.path.join(path, csv_files[0]))
    print(f"Dataset Loaded: {df.shape}")
    print(f"Class distribution:\n{df['Result'].value_counts()}")
    print(f"-1: Legitimate, 1: Phishing")
    return df


# ============================
# 2. ENHANCED MODEL TRAINING WITH BALANCED WEIGHTS
# ============================

def train_balanced_model(df):
    """Train ensemble model with class weight balancing to reduce false negatives"""
    print("\n" + "="*50)
    print("TRAINING BALANCED MODEL (Reducing False Negatives)")
    print("="*50)

    # Prepare features and target
    X = df.drop(["Result", "id"], axis=1)  # Exclude 'id' column
    y = (df["Result"] == 1).astype(int)    # Convert {-1,1} → {0,1} where 1=Phishing

    print(f"Features: {X.shape[1]}")
    print(f"Samples: {X.shape[0]}")
    print(f"Phishing samples: {y.sum()} ({y.sum()/len(y)*100:.1f}%)")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Calculate class weights to reduce false negatives
    # Higher weight for phishing class to reduce false negatives
    phishing_ratio = y_train.sum() / len(y_train)
    legit_ratio = 1 - phishing_ratio

    # Give more weight to phishing class to reduce false negatives
    class_weights = {
        0: 1.0,      # Legitimate
        1: 1.2       # Phishing (20% higher weight)
    }

    # Train Random Forest with balanced class weights
    print("\n[1] Training RandomForest with class weights...")
    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight=class_weights,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2
    )
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)
    print(f"RandomForest Accuracy: {rf_acc:.4f}")

    # Confusion matrix for RF
    rf_cm = confusion_matrix(y_test, rf_pred)
    rf_false_negatives = rf_cm[1, 0] / rf_cm[1, :].sum()
    print(f"RF False Negative Rate: {rf_false_negatives:.2%}")

    # Train XGBoost with scale_pos_weight to reduce false negatives
    print("\n[2] Training XGBoost with scale_pos_weight...")
    # Calculate scale_pos_weight to balance classes
    scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

    xgb = XGBClassifier(
        n_estimators=250,
        max_depth=7,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight * 1.2,  # 20% more weight to phishing
        subsample=0.8,
        colsample_bytree=0.8
    )
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)
    xgb_acc = accuracy_score(y_test, xgb_pred)
    print(f"XGBoost Accuracy: {xgb_acc:.4f}")

    # Confusion matrix for XGB
    xgb_cm = confusion_matrix(y_test, xgb_pred)
    xgb_false_negatives = xgb_cm[1, 0] / xgb_cm[1, :].sum()
    print(f"XGB False Negative Rate: {xgb_false_negatives:.2%}")

    # Train Ensemble with optimized voting weights
    print("\n[3] Training Optimized Ensemble Model...")
    voting = VotingClassifier(
        estimators=[("rf", rf), ("xgb", xgb)],
        voting="soft",
        weights=[1.0, 1.1]  # Give XGB slightly more weight (better at phishing detection)
    )
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)
    voting_acc = accuracy_score(y_test, voting_pred)
    print(f"Ensemble Accuracy: {voting_acc:.4f}")

    # Confusion matrix for Ensemble
    voting_cm = confusion_matrix(y_test, voting_pred)
    voting_false_negatives = voting_cm[1, 0] / voting_cm[1, :].sum()
    voting_false_positives = voting_cm[0, 1] / voting_cm[0, :].sum()
    print(f"Ensemble False Negative Rate: {voting_false_negatives:.2%}")
    print(f"Ensemble False Positive Rate: {voting_false_positives:.2%}")

    # Detailed report
    print("\n" + "="*50)
    print("BALANCED ENSEMBLE MODEL PERFORMANCE")
    print("="*50)
    print(classification_report(y_test, voting_pred,
                               target_names=["Legitimate", "Phishing"]))

    # Confusion Matrix
    print("\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"                Legit  Phishing")
    print(f"Actual Legit     {voting_cm[0,0]:5d}   {voting_cm[0,1]:5d}")
    print(f"Actual Phishing  {voting_cm[1,0]:5d}   {voting_cm[1,1]:5d}")

    # Calculate metrics
    precision = voting_cm[1,1] / (voting_cm[1,1] + voting_cm[0,1])
    recall = voting_cm[1,1] / (voting_cm[1,1] + voting_cm[1,0])
    f1 = 2 * (precision * recall) / (precision + recall)

    print(f"\nPhishing Detection Metrics:")
    print(f"Precision: {precision:.4f} (How many detected phishing are actually phishing)")
    print(f"Recall:    {recall:.4f} (How many actual phishing are detected)")
    print(f"F1-Score:  {f1:.4f} (Balance between precision and recall)")

    return voting, X.columns.tolist()


# ============================
# 3. SAVE/LOAD MODEL
# ============================

def save_model(model, filename="phishing_model_balanced.pkl"):
    """Save trained model to disk"""
    with open(filename, "wb") as f:
        pickle.dump(model, f)
    print(f"\nModel saved as: {filename}")
    return filename

def load_saved_model(filename="phishing_model_balanced.pkl"):
    """Load saved model from disk"""
    with open(filename, "rb") as f:
        model = pickle.load(f)
    print(f"Model loaded from: {filename}")
    return model


# ============================
# 4. ENHANCED FEATURE EXTRACTION WITH BETTER PHISHING DETECTION
# ============================

UCI_FEATURES = [
    "having_IP_Address", "URL_Length", "Shortining_Service", "having_At_Symbol",
    "double_slash_redirecting", "Prefix_Suffix", "having_Sub_Domain",
    "SSLfinal_State", "Domain_registeration_length", "Favicon", "port",
    "HTTPS_token", "Request_URL", "URL_of_Anchor", "Links_in_tags", "SFH",
    "Submitting_to_email", "Abnormal_URL", "Redirect", "on_mouseover",
    "RightClick", "popUpWidnow", "Iframe", "age_of_domain", "DNSRecord",
    "web_traffic", "Page_Rank", "Google_Index", "Links_pointing_to_page",
    "Statistical_report"
]

def extract_features_enhanced(url):
    """
    Enhanced feature extraction with better phishing pattern detection
    """
    features = {}

    # Parse URL
    p = urlparse(url)
    domain = p.netloc
    dom = domain.replace("www.", "").lower()
    path = p.path.lower()
    query = p.query.lower()
    full_url = url.lower()

    # Clean domain for analysis
    if domain.startswith('www.'):
        domain = domain[4:]

    # ========== URL-BASED FEATURES ==========

    # 1. Having IP Address (1=Phishing, -1=Legitimate)
    ip_pattern = r'^(\d{1,3}\.){3}\d{1,3}(:\d+)?$'
    features["having_IP_Address"] = 1 if re.match(ip_pattern, domain) else -1

    # 2. URL Length (1=Phishing for long, -1=Legitimate for short)
    url_length = len(url)
    if url_length < 54:
        features["URL_Length"] = -1  # Legitimate
    elif 54 <= url_length <= 75:
        features["URL_Length"] = 0   # Suspicious
    else:
        features["URL_Length"] = 1   # Phishing

    # 3. Shortening Service - ENHANCED detection
    shorteners = [
        'bit.ly', 'goo.gl', 'tinyurl.com', 'is.gd', 't.co', 'ow.ly', 'buff.ly',
        'adf.ly', 'bitly.com', 'shorte.st', 'tiny.cc', 'short.to', 'tr.im',
        'qr.ae', 'v.gd', 'cur.lv', 'clickmeter.com', 'soo.gd', 'ity.im',
        'q.gs', 'po.st', 'bc.vc', 'twitthis.com', 'u.to', 'j.mp', 'buzurl.com',
        'cutt.us', 'u.bb', 'yourls.org', 'x.co', 'rebrand.ly', 'shink.me',
        'clck.ru', 'cutt.ly', 'shorturl.at', 'rb.gy', 'tiny.cc'
    ]

    is_shortener = any(shortener in domain.lower() for shortener in shorteners)
    features["Shortining_Service"] = 1 if is_shortener else -1

    # 4. Having @ Symbol (1=Phishing, -1=Legitimate)
    features["having_At_Symbol"] = 1 if "@" in url else -1

    # 5. Double Slash Redirecting (1=Phishing, -1=Legitimate)
    path_part = url.split('://', 1)[-1] if '://' in url else url
    features["double_slash_redirecting"] = 1 if path_part.count('//') > 0 else -1

    # 6. Prefix/Suffix (1=Phishing for hyphens, -1=Legitimate)
    features["Prefix_Suffix"] = 1 if "-" in domain else -1

    # 7. Having Sub Domain - ENHANCED counting
    clean_domain = domain.replace('www.', '')
    subdomain_count = clean_domain.count('.')

    if subdomain_count == 0:
        features["having_Sub_Domain"] = -1  # Legitimate
    elif subdomain_count == 1:
        features["having_Sub_Domain"] = 0   # Suspicious
    elif subdomain_count == 2:
        features["having_Sub_Domain"] = 0   # Suspicious (common for country domains)
    else:
        features["having_Sub_Domain"] = 1   # Phishing (too many subdomains)

    # 8. SSL Final State - ENHANCED: HTTPS is LEGITIMATE (-1), HTTP is PHISHING (1)
    if p.scheme == "https":
        features["SSLfinal_State"] = -1  # HTTPS is GOOD
    else:
        features["SSLfinal_State"] = 1   # HTTP is BAD

    # 9. HTTPS Token in Domain (1=Phishing if "https" appears in domain)
    features["HTTPS_token"] = 1 if "https" in domain.lower() else -1

    # 10. Port (1=Phishing for non-standard ports)
    features["port"] = 1 if p.port not in [80, 443, None] else -1

    # ========== ENHANCED PHISHING PATTERN DETECTION ==========

    # Check for suspicious patterns in URL
    suspicious_patterns = [
        'login', 'signin', 'verify', 'secure', 'account', 'update',
        'password', 'banking', 'paypal', 'ebay', 'amazon', 'microsoft',
        'apple', 'netflix', 'facebook', 'instagram', 'whatsapp'
    ]

    # Brand impersonation detection
    brand_in_subdomain = any(brand in domain for brand in ['paypal', 'bank', 'secure', 'login'])
    brand_in_path = any(brand in path for brand in suspicious_patterns)

    # Check for suspicious TLDs
    suspicious_tlds = ['.xyz', '.top', '.club', '.site', '.online', '.info', '.biz']
    has_suspicious_tld = any(tld in domain for tld in suspicious_tlds)

    # Check for IP address in path or query (common in phishing)
    ip_in_path = bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', path))
    ip_in_query = bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', query))

    # ========== PLACEHOLDER FEATURES ==========
    placeholder_features = [
        "Domain_registeration_length", "Favicon", "age_of_domain", "DNSRecord",
        "web_traffic", "Page_Rank", "Google_Index", "Links_pointing_to_page",
        "Statistical_report"
    ]
    for pf in placeholder_features:
        features[pf] = -1  # Default to legitimate

    # ========== HTML-BASED FEATURES ==========
    html_features = [
        "Request_URL", "URL_of_Anchor", "Links_in_tags", "SFH",
        "Submitting_to_email", "Abnormal_URL", "Redirect", "on_mouseover",
        "RightClick", "popUpWidnow", "Iframe"
    ]

    for hf in html_features:
        features[hf] = 0  # Neutral default

    # Try to fetch HTML
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, timeout=5, headers=headers, verify=False)
        response.raise_for_status()
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')

        # Extract HTML elements
        anchors = soup.find_all('a', href=True)
        imgs = soup.find_all('img')
        scripts = soup.find_all('script')
        links = soup.find_all('link')
        iframes = soup.find_all('iframe')
        forms = soup.find_all('form')

        # 11. Request URL (external objects)
        total_objects = len(imgs) + len(scripts)
        if total_objects > 0:
            external_objects = 0
            for tag in imgs + scripts:
                src = tag.get('src', '')
                if src and src.startswith('http') and dom not in src:
                    external_objects += 1
            external_ratio = external_objects / total_objects
            if external_ratio < 0.22:
                features["Request_URL"] = -1
            elif external_ratio <= 0.61:
                features["Request_URL"] = 0
            else:
                features["Request_URL"] = 1
        else:
            features["Request_URL"] = 0

        # 12. URL of Anchor
        if anchors:
            external_anchors = 0
            for a in anchors:
                href = a.get('href', '')
                if href and href.startswith('http') and dom not in href:
                    external_anchors += 1
            external_ratio = external_anchors / len(anchors)
            if external_ratio < 0.31:
                features["URL_of_Anchor"] = -1
            elif external_ratio <= 0.67:
                features["URL_of_Anchor"] = 0
            else:
                features["URL_of_Anchor"] = 1
        else:
            features["URL_of_Anchor"] = 0

        # 13. Links in Tags
        meta_tags = soup.find_all('meta')
        all_tags = scripts + links + meta_tags
        if all_tags:
            external_links = 0
            for tag in all_tags:
                src = tag.get('src', '') or tag.get('href', '') or tag.get('content', '')
                if src and src.startswith('http') and dom not in src:
                    external_links += 1
            external_ratio = external_links / len(all_tags)
            if external_ratio < 0.17:
                features["Links_in_tags"] = -1
            elif external_ratio <= 0.81:
                features["Links_in_tags"] = 0
            else:
                features["Links_in_tags"] = 1
        else:
            features["Links_in_tags"] = 0

        # 14. SFH (Server Form Handler)
        if not forms:
            features["SFH"] = -1
        else:
            form_action = forms[0].get('action', '').lower()
            if not form_action or form_action == 'about:blank':
                features["SFH"] = 1  # Phishing
            elif form_action.startswith('http') and dom not in form_action:
                features["SFH"] = 0  # Suspicious
            else:
                features["SFH"] = -1  # Legitimate

        # 15. Submitting to Email
        features["Submitting_to_email"] = 1 if 'mailto:' in html.lower() else -1

        # 16. Abnormal URL (hostname in page content)
        features["Abnormal_URL"] = 1 if dom not in html.lower() else -1

        # 17. Redirect
        redirect_patterns = [
            'window.location', 'window.open', 'location.href',
            'location.replace', 'http-equiv="refresh"'
        ]
        has_redirect = any(pattern in html.lower() for pattern in redirect_patterns)
        features["Redirect"] = 1 if has_redirect else -1

        # 18. On Mouse Over
        features["on_mouseover"] = 1 if 'onmouseover' in html.lower() else -1

        # 19. Right Click Disabled
        features["RightClick"] = 1 if 'event.button==2' in html or 'contextmenu' in html.lower() else -1

        # 20. Popup Window
        features["popUpWidnow"] = 1 if 'window.open' in html or 'alert(' in html else -1

        # 21. IFrame
        features["Iframe"] = 1 if iframes else -1

    except Exception as e:
        # HTML fetch failed - keep default values
        pass

    # ========== ADDITIONAL ENHANCED FEATURES ==========

    # Brand impersonation score
    brand_impersonation_score = 0
    if brand_in_subdomain:
        brand_impersonation_score += 2
    if brand_in_path:
        brand_impersonation_score += 2
    if has_suspicious_tld:
        brand_impersonation_score += 1
    if ip_in_path or ip_in_query:
        brand_impersonation_score += 2

    # If brand impersonation score is high, mark some features as more suspicious
    if brand_impersonation_score >= 3:
        if features["having_Sub_Domain"] == -1:
            features["having_Sub_Domain"] = 0
        if features["URL_Length"] == -1:
            features["URL_Length"] = 0

    # Ensure all features are present
    for feature in UCI_FEATURES:
        if feature not in features:
            features[feature] = 0

    return features


# ============================
# 5. IMPROVED PREDICTION WITH BETTER BALANCE
# ============================

def predict_url_balanced(url, model, feature_cols):
    """
    Improved prediction with better balance between false positives and false negatives
    """
    # Extract features with enhanced logic
    features = extract_features_enhanced(url)

    # Create dataframe for model prediction
    df = pd.DataFrame([features])[feature_cols]

    # Get model prediction
    prob = model.predict_proba(df)[0][1]  # Probability of being phishing
    pred = model.predict(df)[0]  # 1=Phishing, 0=Legitimate

    # Parse URL for additional checks
    p = urlparse(url)
    domain = p.netloc.lower()
    path = p.path.lower()

    # ========== ENHANCED RULE-BASED ADJUSTMENTS ==========

    # WHITELIST: Known legitimate domains (exact or partial matching)
    legit_whitelist = [
        'google.com', 'github.com', 'amazon.com', 'wikipedia.org',
        'microsoft.com', 'apple.com', 'netflix.com', 'twitter.com',
        'facebook.com', 'instagram.com', 'linkedin.com', 'chatgpt.com',
        'openai.com', 'deepseek.com', 'anthropic.com', 'yahoo.com',
        'reddit.com', 'stackoverflow.com', 'medium.com', 'quora.com',
        'cloudflare.com', 'mozilla.org', 'python.org', 'w3.org'
    ]

    # Check if domain is in whitelist (exact match after removing www.)
    clean_domain = domain.replace('www.', '')
    is_whitelisted = any(clean_domain == legit_domain or
                        clean_domain.endswith('.' + legit_domain)
                        for legit_domain in legit_whitelist)

    if is_whitelisted:
        # Known legitimate site - reduce probability but don't override completely
        prob = max(0.05, prob * 0.3)  # Reduce probability significantly
        if prob < 0.3:  # Only override if confidence is low
            pred = 0
        return pred, prob, features

    # BLACKLIST: Strong phishing patterns
    phishing_blacklist_patterns = [
        'paypal-security', 'bank-security', 'login-verify',
        'account-update', 'secure-verify', 'password-reset',
        'update-account', 'verify-login', 'security-alert'
    ]

    # Suspicious TLDs
    suspicious_tlds = ['.xyz', '.top', '.club', '.site', '.online', '.info', '.biz', '.ru', '.cn']

    # Check for multiple strong indicators
    strong_indicators = []

    # 1. URL Shortener
    if features["Shortining_Service"] == 1:
        strong_indicators.append("URL Shortener")

    # 2. IP Address in URL
    if features["having_IP_Address"] == 1:
        strong_indicators.append("IP Address in URL")

    # 3. HTTP (not HTTPS)
    if features["SSLfinal_State"] == 1 and ('login' in path or 'signin' in path or 'bank' in path):
        strong_indicators.append("HTTP with sensitive page")

    # 4. Suspicious TLD
    if any(tld in domain for tld in suspicious_tlds):
        strong_indicators.append("Suspicious TLD")

    # 5. Brand in subdomain but not legitimate brand
    suspicious_brands_in_subdomain = ['paypal', 'bank', 'secure', 'login', 'verify', 'account']
    if any(brand in domain for brand in suspicious_brands_in_subdomain):
        # Check if it's actually the legitimate brand
        if not ('paypal.com' in domain or 'bankofamerica.com' in domain):
            strong_indicators.append("Brand impersonation in subdomain")

    # 6. Too many subdomains
    if features["having_Sub_Domain"] == 1 and domain.count('.') > 3:
        strong_indicators.append("Too many subdomains")

    # Rule-based adjustments based on strong indicators
    if len(strong_indicators) >= 2:
        # Multiple strong indicators = very likely phishing
        pred = 1
        prob = max(prob, 0.85)
    elif len(strong_indicators) == 1:
        # Single strong indicator = increase probability
        prob = min(0.95, prob * 1.3)
        if prob > 0.7:
            pred = 1

    # Adjust for borderline cases
    if 0.4 <= prob <= 0.6:
        # Borderline case - check additional features
        legit_features_count = sum(1 for k, v in features.items() if v == -1)
        phishing_features_count = sum(1 for k, v in features.items() if v == 1)

        if legit_features_count > phishing_features_count + 5:
            # More legitimate features
            pred = 0
            prob = prob * 0.7
        elif phishing_features_count > legit_features_count + 5:
            # More phishing features
            pred = 1
            prob = prob * 1.3

    # Final confidence adjustment
    if pred == 1 and prob < 0.5:
        prob = 0.5  # Minimum confidence for phishing prediction
    elif pred == 0 and prob > 0.5:
        prob = 0.5  # Maximum confidence for legitimate prediction

    return pred, prob, features


# ============================
# 6. COMPREHENSIVE TESTING WITH BALANCE ANALYSIS
# ============================

def test_with_balance_analysis(urls, model, feature_cols):
    """Test URLs with detailed balance analysis"""
    print("\n" + "="*60)
    print("BALANCE ANALYSIS TESTING")
    print("="*60)

    results = []

    # Define expected labels for each URL
    expected_labels = {
        # Legitimate
        "https://google.com": 0,
        "https://github.com": 0,
        "https://www.amazon.com": 0,
        "https://www.wikipedia.org": 0,
        "https://chatgpt.com": 0,
        "https://chat.deepseek.com": 0,
        "https://www.microsoft.com": 0,
        "https://www.apple.com": 0,

        # Phishing (these were false negatives)
        "http://192.168.1.1/login.php": 1,
        "http://paypal-secure-verify.xyz/login": 1,
        "http://bit.ly/2fSdq": 1,
        "http://free-gift-cards-now.com/claim": 1,
        "https://www.paypal.com.us.security.verify-account.com": 1,
        "http://198.54.23.11/login/update": 1,

        # Edge cases
        "http://example.com/login": 0,  # Should be legitimate but HTTP
        "https://secure-bank-login.com": 1,  # Suspicious
        "https://paypal-security-alert.com/verify": 1,  # Phishing
        "http://192.168.1.100:8080/admin": 1,  # Internal IP with port
    }

    for url in urls:
        pred, prob, features = predict_url_balanced(url, model, feature_cols)
        expected = expected_labels.get(url, -1)

        # Determine if prediction is correct
        if expected != -1:
            is_correct = (pred == expected)
            error_type = None
            if not is_correct:
                if pred == 1 and expected == 0:
                    error_type = "False Positive"
                elif pred == 0 and expected == 1:
                    error_type = "False Negative"
        else:
            is_correct = None
            error_type = "Unknown expected"

        results.append({
            'url': url,
            'prediction': 'Phishing' if pred == 1 else 'Legitimate',
            'confidence': prob,
            'expected': 'Phishing' if expected == 1 else 'Legitimate' if expected == 0 else 'Unknown',
            'correct': is_correct,
            'error_type': error_type,
            'phishing_indicators': sum(1 for v in features.values() if v == 1),
            'legitimate_indicators': sum(1 for v in features.values() if v == -1)
        })

        # Print result
        print(f"\n{'='*40}")
        print(f"URL: {url[:50]}{'...' if len(url) > 50 else ''}")
        print(f"Prediction: {'🚨 PHISHING' if pred == 1 else '✅ LEGITIMATE'}")
        print(f"Confidence: {prob:.1%}")
        if expected != -1:
            print(f"Expected: {'🚨 PHISHING' if expected == 1 else '✅ LEGITIMATE'}")
            if is_correct:
                print(f"Status: ✅ CORRECT")
            else:
                print(f"Status: ❌ {error_type}")
        print(f"Phishing indicators: {results[-1]['phishing_indicators']}")
        print(f"Legitimate indicators: {results[-1]['legitimate_indicators']}")

    return results


# ============================
# 7. MAIN EXECUTION WITH BALANCE FOCUS
# ============================

if __name__ == "__main__":
    print("="*70)
    print("BALANCED PHISHING DETECTOR (Reducing False Negatives)")
    print("="*70)

    # Option 1: Train new balanced model
    train_new = input("\nTrain new balanced model? (y/n): ").lower().strip() == 'y'

    if train_new:
        # Load dataset and train
        df = load_dataset()
        model, feature_cols = train_balanced_model(df)

        # Save model
        model_file = save_model(model, "phishing_detector_balanced.pkl")

        # Save feature columns
        with open("feature_columns_balanced.pkl", "wb") as f:
            pickle.dump(feature_cols, f)
        print("Feature columns saved.")

    else:
        # Load existing model
        try:
            model = load_saved_model("phishing_detector_balanced.pkl")
            with open("feature_columns_balanced.pkl", "rb") as f:
                feature_cols = pickle.load(f)
            print(f"Loaded {len(feature_cols)} features")
        except FileNotFoundError:
            print("No saved model found. Training new balanced model...")
            df = load_dataset()
            model, feature_cols = train_balanced_model(df)
            save_model(model, "phishing_detector_balanced.pkl")
            with open("feature_columns_balanced.pkl", "wb") as f:
                pickle.dump(feature_cols, f)

    # Test URLs
    test_urls_list = [
        # Legitimate sites
        "https://google.com",
        "https://github.com",
        "https://www.amazon.com",
        "https://www.wikipedia.org",
        "https://chatgpt.com",
        "https://chat.deepseek.com",
        "https://www.microsoft.com",
        "https://www.apple.com",

        # Phishing sites (previously false negatives)
        "http://192.168.1.1/login.php",
        "http://paypal-secure-verify.xyz/login",
        "http://bit.ly/2fSdq",
        "http://free-gift-cards-now.com/claim",
        "https://www.paypal.com.us.security.verify-account.com",
        "http://198.54.23.11/login/update",

        # Edge cases
        "http://example.com/login",  # HTTP login page
        "https://secure-bank-login.com",  # Looks suspicious
        "https://paypal-security-alert.com/verify",  # Brand impersonation
        "http://192.168.1.100:8080/admin",  # Internal IP with port
    ]

    # Test with balance analysis
    results = test_with_balance_analysis(test_urls_list, model, feature_cols)

    # Calculate metrics
    print("\n" + "="*70)
    print("PERFORMANCE METRICS")
    print("="*70)

    total = len(results)
    correct = sum(1 for r in results if r['correct'] is True)
    false_positives = sum(1 for r in results if r['error_type'] == 'False Positive')
    false_negatives = sum(1 for r in results if r['error_type'] == 'False Negative')

    accuracy = correct / total if total > 0 else 0

    print(f"\nTotal URLs: {total}")
    print(f"Correct predictions: {correct} ({accuracy:.1%})")
    print(f"False Positives: {false_positives}")
    print(f"False Negatives: {false_negatives}")

    # Show details of errors
    if false_positives > 0:
        print(f"\n🔍 False Positives (Legitimate marked as Phishing):")
        for r in results:
            if r['error_type'] == 'False Positive':
                print(f"  - {r['url'][:60]}...")
                print(f"    Confidence: {r['confidence']:.1%}, Phishing indicators: {r['phishing_indicators']}")

    if false_negatives > 0:
        print(f"\n🔍 False Negatives (Phishing marked as Legitimate):")
        for r in results:
            if r['error_type'] == 'False Negative':
                print(f"  - {r['url'][:60]}...")
                print(f"    Confidence: {r['confidence']:.1%}, Legitimate indicators: {r['legitimate_indicators']}")

    # Interactive mode with better guidance
    while True:
        print("\n" + "-"*50)
        user_url = input("\nEnter URL to check (or 'quit' to exit): ").strip()

        if user_url.lower() in ['quit', 'exit', 'q']:
            print("Exiting...")
            break

        if not user_url.startswith(('http://', 'https://')):
            user_url = 'https://' + user_url

        try:
            pred, prob, features = predict_url_balanced(user_url, model, feature_cols)

            print(f"\n{'='*60}")
            print(f"RESULT: {'🚨 PHISHING' if pred == 1 else '✅ LEGITIMATE'}")
            print(f"Confidence: {prob:.1%}")
            print(f"{'='*60}")

            # Detailed analysis
            p = urlparse(user_url)
            print(f"\n🔍 URL Analysis:")
            print(f"  Domain: {p.netloc}")
            print(f"  Protocol: {p.scheme} ({'🔒 Secure' if p.scheme == 'https' else '⚠️  Insecure'})")
            print(f"  Path: {p.path[:50]}{'...' if len(p.path) > 50 else ''}")

            if p.query:
                print(f"  Query parameters: {len(p.query.split('&'))} parameters")

            # Key risk factors
            print(f"\n📊 Risk Assessment:")

            high_risk_factors = []
            medium_risk_factors = []

            # High risk factors
            if features.get("having_IP_Address") == 1:
                high_risk_factors.append("IP address in domain")
            if features.get("Shortining_Service") == 1:
                high_risk_factors.append("URL shortening service")
            if features.get("having_At_Symbol") == 1:
                high_risk_factors.append("@ symbol in URL")
            if features.get("SSLfinal_State") == 1 and ('login' in p.path or 'signin' in p.path):
                high_risk_factors.append("HTTP protocol on login page")

            # Medium risk factors
            if features.get("Prefix_Suffix") == 1:
                medium_risk_factors.append("Hyphen in domain name")
            if features.get("having_Sub_Domain") == 1 and p.netloc.count('.') > 3:
                medium_risk_factors.append("Too many subdomains")
            if features.get("URL_Length") == 1:
                medium_risk_factors.append("Very long URL")
            if features.get("popUpWidnow") == 1:
                medium_risk_factors.append("Popup windows detected")
            if features.get("Iframe") == 1:
                medium_risk_factors.append("IFrames detected")

            # Display risk factors
            if high_risk_factors:
                print(f"  🚨 High Risk Factors ({len(high_risk_factors)}):")
                for factor in high_risk_factors:
                    print(f"     • {factor}")

            if medium_risk_factors:
                print(f"  ⚠️  Medium Risk Factors ({len(medium_risk_factors)}):")
                for factor in medium_risk_factors[:3]:  # Show only top 3
                    print(f"     • {factor}")
                if len(medium_risk_factors) > 3:
                    print(f"     • ... and {len(medium_risk_factors) - 3} more")

            if not high_risk_factors and not medium_risk_factors:
                print(f"  ✅ No significant risk factors detected")

            # Recommendations
            print(f"\n💡 Recommendations:")
            if pred == 1:
                print(f"  • Do not enter any personal information")
                print(f"  • Do not download any files from this site")
                print(f"  • Verify the URL with the official website")
                if p.scheme == 'http':
                    print(f"  • This site uses HTTP (not secure HTTPS)")
            else:
                if p.scheme == 'http':
                    print(f"  • This site uses HTTP instead of HTTPS")
                    print(f"  • Consider using the HTTPS version if available")
                print(f"  • Always check for the padlock icon in your browser")

            # Ask for detailed feature analysis
            if input("\nShow detailed feature analysis? (y/n): ").lower() == 'y':
                phishing_count = sum(1 for v in features.values() if v == 1)
                legit_count = sum(1 for v in features.values() if v == -1)
                suspicious_count = sum(1 for v in features.values() if v == 0)

                print(f"\nDetailed Feature Count:")
                print(f"  🚨 Phishing indicators: {phishing_count}")
                print(f"  ⚠️  Suspicious indicators: {suspicious_count}")
                print(f"  ✅ Legitimate indicators: {legit_count}")

                # Show top indicators
                if phishing_count > 0:
                    print(f"\nTop phishing indicators:")
                    for k, v in sorted(features.items(), key=lambda x: x[1], reverse=True):
                        if v == 1 and k in ["having_IP_Address", "Shortining_Service",
                                          "having_At_Symbol", "SSLfinal_State", "Prefix_Suffix"]:
                            print(f"  • {k.replace('_', ' ').title()}")

        except Exception as e:
            print(f"Error analyzing URL: {e}")
            print("Please check the URL format and try again.")

BALANCED PHISHING DETECTOR (Reducing False Negatives)


100%|██████████| 110k/110k [00:00<00:00, 30.0MB/s]

Extracting files...
Dataset Loaded: (11055, 32)
Class distribution:
Result
 1    6157
-1    4898
Name: count, dtype: int64
-1: Legitimate, 1: Phishing

TRAINING BALANCED MODEL (Reducing False Negatives)
Features: 30
Samples: 11055
Phishing samples: 6157 (55.7%)



[1] Training RandomForest with class weights...
RandomForest Accuracy: 0.9643
RF False Negative Rate: 2.03%

[2] Training XGBoost with scale_pos_weight...
XGBoost Accuracy: 0.9765
XGB False Negative Rate: 1.38%

[3] Training Optimized Ensemble Model...
Ensemble Accuracy: 0.9738
Ensemble False Negative Rate: 1.30%
Ensemble False Positive Rate: 4.29%

BALANCED ENSEMBLE MODEL PERFORMANCE
              precision    recall  f1-score   support

  Legitimate       0.98      0.96      0.97       980
    Phishing       0.97      0.99      0.98      1231

    accuracy                           0.97      2211
   macro avg       0.97      0.97      0.97      2211
weighted avg       0.97      0.97      0.97      2211


Confusion Matrix:
                Predicted
                Legit  Phishing
Actual Legit       938      42
Actual Phishing     16    1215

Phishing Detection Metrics:
Precision: 0.9666 (How many detected phishing are actually phishing)
Recall:    0.9870 (How many actual phishing are